## Imports and Configuration

In [4]:
import pandas as pd
from sqlalchemy import create_engine, inspect
from snowflake.sqlalchemy import URL
from urllib.parse import quote_plus

# --- POSTGRES CONFIG ---
PG_DB = {
    "user": "postgres",
    "pass": "Malli@452",
    "host": "localhost",
    "port": "5432",
    "db": "malli_db"
}

# --- SNOWFLAKE CONFIG ---
SNOW_CONF = {
    "account": "KWEQZLG-IF45305", 
    "user": "CHINNA",
    "password": "Chinnamaya@123",
    "database": "ELT_DEMO",
    "schema": "ELT_SCHEMA",
    "warehouse": "COMPUTE_WH",
    "role": "ACCOUNTADMIN"
}

print("Libraries imported and config set")


Libraries imported and config set


## Create Engines and Test Connections

In [5]:

# Postgres Engine
pg_pass = quote_plus(PG_DB['pass'])
pg_url = f"postgresql://{PG_DB['user']}:{pg_pass}@{PG_DB['host']}:{PG_DB['port']}/{PG_DB['db']}"
pg_engine = create_engine(pg_url)

# Snowflake Engine
snow_url = URL(
    account=SNOW_CONF['account'],
    user=SNOW_CONF['user'],
    password=SNOW_CONF['password'],
    database=SNOW_CONF['database'],
    schema=SNOW_CONF['schema'],
    warehouse=SNOW_CONF['warehouse'],
    role=SNOW_CONF['role']
)
snow_engine = create_engine(snow_url)

# Test
try:
    with pg_engine.connect() as conn:
        print("Postgres Connection Successful")
    with snow_engine.connect() as conn:
        print("Snowflake Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

Postgres Connection Successful
Snowflake Connection Successful


## Inspect Tables

In [6]:
inspector = inspect(pg_engine)
pg_tables = inspector.get_table_names(schema='public') # Pulling from dvdrental public schema in postgresql

print(f"Found {len(pg_tables)} tables in SQL Server: {pg_tables}")

Found 12 tables in SQL Server: ['famous', 'sf_transactions', 'users', 'friends', 'ms_projects', 'ms_emp_projects', 'airbnb_apartments', 'airbnb_hosts', 'user_purchases', 'car_launches', 'nominee_information', 'oscar_nominees']


## The Migration Loop

In [7]:
for table in pg_tables:
    try:
        if table == 'film':
            print(f"Skipping {table} (Excluded intentionally)")
            continue

        print(f"--- Migrating: {table} to Snowflake ---")
        
        # EXTRACT
        df = pd.read_sql_table(table, pg_engine, schema='public')

        if df.empty:
            print(f"Table {table} is empty. Skipping...")
            continue
        
        # TRANSFORM: Prepend prefix (Snowflake will uppercase this)
        destination_name = f"stg_{table}"
        
        # LOAD
        # Snowflake to_sql is slightly slower than local DBs because it's cloud-based.
        # For huge datasets, Snowflake prefers "Stage & Copy" via S3, but for 
        # small/medium tables, to_sql works fine.
        print(f"Writing {table} to Snowflake ({len(df)} rows)...")
        df.to_sql(
            destination_name, 
            snow_engine, 
            if_exists='replace', 
            index=False,
            chunksize=5000 # Snowflake handles large chunks well
        )
        print(f"Successfully moved {table} -> {destination_name.upper()}")
        
    except Exception as e:
        print(f"Error with {table}: {e}")

--- Migrating: famous to Snowflake ---
Writing famous to Snowflake (13 rows)...
Successfully moved famous -> STG_FAMOUS
--- Migrating: sf_transactions to Snowflake ---
Writing sf_transactions to Snowflake (23 rows)...
Successfully moved sf_transactions -> STG_SF_TRANSACTIONS
--- Migrating: users to Snowflake ---
Writing users to Snowflake (10 rows)...
Successfully moved users -> STG_USERS
--- Migrating: friends to Snowflake ---
Writing friends to Snowflake (15 rows)...
Successfully moved friends -> STG_FRIENDS
--- Migrating: ms_projects to Snowflake ---
Writing ms_projects to Snowflake (20 rows)...
Successfully moved ms_projects -> STG_MS_PROJECTS
--- Migrating: ms_emp_projects to Snowflake ---
Writing ms_emp_projects to Snowflake (20 rows)...
Successfully moved ms_emp_projects -> STG_MS_EMP_PROJECTS
--- Migrating: airbnb_apartments to Snowflake ---
Writing airbnb_apartments to Snowflake (9 rows)...
Successfully moved airbnb_apartments -> STG_AIRBNB_APARTMENTS
--- Migrating: airbnb_hos